In [4]:
import pandas as pd
import numpy as np

# Load data
df_topic = pd.read_csv("hasil_topic_assignment_automerged.csv")
df_seg = pd.read_csv("sampled_segmentation.csv")

# Standarisasi nama game agar cocok
df_seg["Game Name"] = df_seg["Game Name"].str.strip().str.lower()
df_topic["game"] = df_topic["game"].str.strip().str.lower()

# Merge berdasarkan nama game
merged = pd.merge(df_seg, df_topic, left_on="Game Name", right_on="game", how="inner")

# Hapus game tidak valid dan nonaktif
merged = merged[merged["Game Name"] != "unknown"]
merged = merged[merged["Playtime (hours)"] >= 0.167]  # 10 menit

# Hapus NaN dan duplikat
merged.dropna(subset=["Steam ID", "Game Name", "Playtime (hours)", "Genres", "Achievements", "topic"], inplace=True)
merged.drop_duplicates(subset=["Steam ID", "Game Name"], inplace=True)

# Hitung total game per pemain
game_counts = merged.groupby("Steam ID")["Game Name"].nunique().reset_index(name="Total_Games")
q1_games = game_counts["Total_Games"].quantile(0.25)
active_players = game_counts[game_counts["Total_Games"] >= q1_games]["Steam ID"]

# Filter hanya pemain aktif
merged = merged[merged["Steam ID"].isin(active_players)]

# Hitung total achievements per pemain
achievement_sum = merged.groupby("Steam ID")["Achievements"].sum().reset_index(name="Total_Achievements")

# Mapping genre ke kode numerik
merged["Genres"] = merged["Genres"].astype(str)
genre_list = sorted(set(g for genres in merged["Genres"] for g in genres.split(", ")))
genre_code_map = {g: i+1 for i, g in enumerate(genre_list)}

# Ambil genre pertama sebagai representasi dominan untuk penyederhanaan
merged["Dominant_Genre_Code"] = merged["Genres"].apply(lambda x: genre_code_map[x.split(", ")[0]] if x else np.nan)

# Hitung topik dominan per pemain
dominant_topic = merged.groupby(["Steam ID", "topic"]).size().reset_index(name="count")
dominant_topic = dominant_topic.sort_values(["Steam ID", "count"], ascending=[True, False])
dominant_topic = dominant_topic.drop_duplicates(subset=["Steam ID"], keep="first")
dominant_topic.rename(columns={"topic": "Dominant_Topic"}, inplace=True)

# Hitung genre dominan per pemain
dominant_genre = merged.groupby(["Steam ID", "Dominant_Genre_Code"]).size().reset_index(name="count")
dominant_genre = dominant_genre.sort_values(["Steam ID", "count"], ascending=[True, False])
dominant_genre = dominant_genre.drop_duplicates(subset=["Steam ID"], keep="first")

# Gabungkan hasil
df_final = pd.merge(game_counts, achievement_sum, on="Steam ID")
df_final = pd.merge(df_final, dominant_topic[["Steam ID", "Dominant_Topic"]], on="Steam ID")
df_final = pd.merge(df_final, dominant_genre[["Steam ID", "Dominant_Genre_Code"]], on="Steam ID")

# Simpan hasil akhir
df_final.to_csv("transformation_segmentation.csv", index=False)
print("✅ Data preprocessing selesai. Hasil disimpan di 'transformation_segmentation.csv'")


✅ Data preprocessing selesai. Hasil disimpan di 'transformation_segmentation.csv'


In [5]:
# Simpan mapping genre dan kode
genre_df = pd.DataFrame(list(genre_code_map.items()), columns=["Genre", "Genre_Code"])
genre_df.to_csv("genre_code_mapping.csv", index=False)
print("✅ Mapping kode genre disimpan di 'genre_code_mapping.csv'")


✅ Mapping kode genre disimpan di 'genre_code_mapping.csv'
